**Runtime → Change runtime type → GPU (T4 or better)** before running.

Research prototype — results are not clinically validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/GlucoIris'
import os; os.makedirs(f'{BASE}/weights', exist_ok=True); os.makedirs(f'{BASE}/reports', exist_ok=True)

In [ ]:
!pip -q install timm
# Code bundle: upload ml/colab/glucoiris_ml_code.zip to MyDrive/GlucoIris/ first
!rm -rf /content/ml && unzip -q -o "$BASE/glucoiris_ml_code.zip" -d /content/
!ls /content/ml
!nvidia-smi --query-gpu=name,memory.total --format=csv

## APTOS 2019 data
Upload your `kaggle.json` (Kaggle → Account → Create API token) and accept the competition rules at kaggle.com/c/aptos2019-blindness-detection first.

In [ ]:
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    up = files.upload()  # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install kaggle
!kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos_zip
!mkdir -p /content/aptos && unzip -q -o /content/aptos_zip/aptos2019-blindness-detection.zip -d /content/aptos
!ls /content/aptos && head -3 /content/aptos/train.csv

## Train the five-class ViT
Class balancing (weighted sampler + class-weighted loss), augmentation, best-QWK selection, temperature scaling and a validation-chosen referable threshold are all inside `train_vit.py`.

In [ ]:
!cd /content && python ml/train_vit.py --dataset aptos --data /content/aptos \
    --model vit_base_patch16_224.augreg_in21k_ft_in1k --epochs 15 --batch 32 --workers 4 \
    --cache /content/aptos_cache --out "$BASE/weights/vit_dr5.pt"

In [ ]:
import json, glob
rep = json.load(open('/content/ml/reports/vit_aptos.json')) if os.path.exists('/content/ml/reports/vit_aptos.json') else json.load(open(glob.glob(f'{BASE}/reports/vit_aptos.json')[0]))
m = rep['meta']['test_metrics']
print('TEST accuracy', round(m['accuracy'],4), 'macro-F1', round(m['macro_f1'],4), 'QWK', round(m['qwk'],4))
print('Referable sens/spec', round(m['referable']['sensitivity'],4), round(m['referable']['specificity'],4), '@', m['referable']['threshold'])
print('Confusion matrix', m['confusion_matrix'])
!cp /content/ml/reports/vit_aptos.json "$BASE/reports/" 2>/dev/null; ls -la "$BASE/weights"

Download `MyDrive/GlucoIris/weights/vit_dr5.pt` and place it in the project at `ml/weights/vit_dr5.pt`, then restart the backend.